In [ ]:
# installs for kaggle env difficulties 

!apt-get update && apt-get install -y swig build-essential
!pip install --upgrade setuptools wheel
!pip install --no-build-isolation gymnasium[box2d] stable-baselines3[extra]

In [ ]:
import gymnasium as gym
from tqdm import tqdm 
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import os
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.logger import configure
from stable_baselines3.common.vec_env import SubprocVecEnv
import pandas as pd 
import torch

In [ ]:
log_dir = "./racing/logs/"
model_log_dir = './racing/model_logs/'
results_dir = "./racing/results/"

num_timesteps = 2048

print("CPU logical cores:", os.cpu_count())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Torch threads:", torch.get_num_threads())

## Train through PPO

In [ ]:
def train(): 
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_log_dir, exist_ok=True)

    logger = configure(model_log_dir, ["stdout", "csv", "tensorboard"])
    
    vec_env = make_vec_env(
        "CarRacing-v3",
        monitor_dir=log_dir,
        n_envs=4, 
        seed=42, 
        vec_env_cls=SubprocVecEnv, 
        env_kwargs={"continuous": False, "render_mode": None}
    )
    
    model = PPO(
        "CnnPolicy",
        vec_env,
        verbose=1,
        seed=42, 
        device = "cuda" if torch.cuda.is_available() else "cpu"
    )

    # from stable_baselines3.common.callbacks import CheckpointCallback
    # Save a checkpoint every 1000 steps
    # checkpoint_callback = CheckpointCallback(
    #     save_freq=1000,
    #     save_path="./logs/",
    #     name_prefix="rl_model",
    #     save_replay_buffer=True,
    #     save_vecnormalize=True,
    # )

    # model.learn(total_timesteps=num_timesteps, callback=checkpoint_callback)

    model.set_logger(logger)
    
    model.learn(total_timesteps=num_timesteps)
    model.save("ppo_racing")
    vec_env.close()
    print("finished training")


train()

In [ ]:
## Plotting training logs 

In [ ]:
from stable_baselines3.common.results_plotter import plot_results
from stable_baselines3.common import results_plotter

os.makedirs(results_dir, exist_ok=True)

plot_results([log_dir], None, results_plotter.X_TIMESTEPS, "PPO Car - Rewards vs. Timestep")

plot_results([log_dir], None, results_plotter.X_EPISODES, "Rewards vs. Episode")

plot_results([log_dir], None, results_plotter.X_WALLTIME, "Rewards vs. Time")


## Test trained model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


model = PPO.load("ppo_racing")

eval_env = gym.make(
    "CarRacing-v3",
    continuous=False,
    render_mode=None
)

total_rewards = []
completion_results = []

num_episodes = 5

for episode in range(num_episodes):

    print("episode: ", episode)
    obs, info = eval_env.reset()

    episode_reward = 0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)

        obs, reward, terminated, truncated, info = eval_env.step(action)

        episode_reward += reward
        done = terminated or truncated

    total_rewards.append(episode_reward)

    completion_results.append(episode_reward > 0)

eval_env.close()

print(f"Average Reward: {np.mean(total_rewards):.3f}")
print(f"Standard Deviation: {np.std(total_rewards):.3f}")
print(f"Completion Rate (reward proxy): {np.mean(completion_results):.1%}")

episodes = np.arange(1, num_episodes + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    episodes,
    total_rewards,
    marker="o",
    label="Episode Reward"
)

plt.axhline(
    np.mean(total_rewards),
    linestyle="--",
    label=f"Mean Reward: {np.mean(total_rewards):.2f}"
)

plt.xlabel("Evaluation Episode")
plt.ylabel("Total Episode Reward")
plt.title("PPO CarRacing: Evaluation Rewards")
plt.xticks(episodes)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()